# Tools — 도구를 "만드는" 법 (데스크탑 Ollama판)

코스 섹션 [Tools](https://huggingface.co/learn/agents-course/unit2/smolagents/tools)를 내 데스크탑 Ollama(`qwen3.5:9b`)로 해 봐요.
지난 두 섹션은 "도구가 주어졌을 때 **어떻게 부르나**"였고, 이번엔 "도구를 **어떻게 만드나**"예요.

**한 문장 요약: 도구 = 파이썬 함수 + 모델이 읽는 사용 설명서(이름·설명·입력·출력)**

| 단계 | 하는 일 | 모델 필요? |
|---|---|---|
| 0~1 | 준비 확인, 모델 연결 | 1단계부터 |
| 2 | `@tool`로 도구 만들기 → **설명서 4칸** 꺼내 보기, 모델 없이 직접 불러 보기 | ❌ |
| 3 | 그 도구를 에이전트에게 주기 (원문 예제 ①) | ✅ |
| 4 | 설명서를 대충 쓰면? → smolagents가 **만들 때부터** 막음 | ❌ |
| 5 | `Tool` 클래스로 도구 만들기 (원문 예제 ②) + 목록에 없는 값 넣어 보기 | ✅ |
| 6 | 같은 도구를 두 방법으로 → 모델이 보는 설명서가 **똑같은지** | ❌ |
| 7 | 기본 도구 상자 (`add_base_tools`) + 에이전트가 **나에게 질문**하는 도구 | ✅ |
| 8 | Hub·Space·LangChain·MCP (읽기만, 실행 안 함) | — |

같은 venv(`units\unit2-frameworks\code\.venv`)를 써요. 새로 설치할 건 없어요.
❌ 표시 칸은 모델을 안 불러서 **바로** 끝나요.

## 0단계: 준비 확인

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)
for pkg in ["litellm", "ddgs", "markdownify", "requests"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)

✅ **확인할 것**: `smolagents 버전: 1.26.0`, 나머지 전부 `True`
- `ModuleNotFoundError` → venv를 켜지 않고 Jupyter를 열었음
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음

## 1단계: 모델 연결

지난 노트북들과 **같은 설정**이에요.

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

reply = model([{"role": "user", "content": [{"type": "text", "text": "Say hello in one short sentence."}]}])
print(reply.content)

✅ **확인할 것**: 짧은 인사말

## 2단계: `@tool`로 도구 만들기 — 설명서 4칸 꺼내 보기

원문 예제 ①의 출장 요리 도구예요. 평범한 파이썬 함수 위에 **`@tool` 한 줄**만 붙였어요.

smolagents는 이 함수에서 모델이 읽을 **설명서 4칸**을 자동으로 뽑아요.

| 설명서 칸 | 어디서 뽑나 | 이 예제에서 |
|---|---|---|
| 이름 (`name`) | 함수 이름 | `catering_service_tool` |
| 설명 (`description`) | docstring 첫 부분 | "This tool returns the highest-rated ..." |
| 입력 (`inputs`) | 타입 힌트 `query: str` + docstring `Args:` 줄 | `query` (string): "A search term ..." |
| 출력 (`output_type`) | 타입 힌트 `-> str` | string |

이 칸은 **모델 없이** 실행돼요.

In [ ]:
from smolagents import tool


@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # 가짜 데이터: 출장 요리 업체와 별점
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    # 별점이 가장 높은 업체 고르기 (query는 안 씀 → 검색하는 척하는 연습용 도구)
    best_service = max(services, key=services.get)
    return best_service


print("① 이름 :", catering_service_tool.name)
print("② 설명 :", catering_service_tool.description)
print("③ 입력 :", catering_service_tool.inputs)
print("④ 출력 :", catering_service_tool.output_type)

print("\n----- 모델(CodeAgent)이 system prompt에서 실제로 읽는 모양 -----")
print(catering_service_tool.to_code_prompt())

print("\n----- 모델 없이 내가 직접 불러 보기 -----")
print(catering_service_tool("anything"))

✅ **확인할 것**
- ①~④가 위 표와 같게 나옴 = 함수 이름·타입 힌트·docstring이 그대로 **설명서**가 됨
- 가운데 칸: 모델은 **함수 속 코드(`services = ...`)를 못 봐요.** `def ...` 한 줄과 설명만 봐요. 그래서 원문이 "이름을 분명하게, docstring을 잘 쓰라"고 하는 거예요
- 마지막 줄 `Gotham Catering Co.` = 도구는 결국 **그냥 함수**라서 모델 없이도 불러 쓸 수 있음

## 3단계: 그 도구를 에이전트에게 주기 (원문 예제 ①)

원문 코드에서 `InferenceClientModel()` → `model`만 바꿨어요.

In [ ]:
from smolagents import CodeAgent

agent = CodeAgent(tools=[catering_service_tool], model=model, max_steps=4)

result = agent.run("Can you give me the name of the highest-rated catering service in Gotham City?")
print("\n★ 최종 답:", result)

✅ **확인할 것**
- 코드 상자 안에 `catering_service_tool(query="...")` 같은 줄 = 모델이 **2단계에서 본 설명서대로** 도구를 부름
- 최종 답 `Gotham Catering Co.`
- `query`에 뭘 넣든 답이 같아요 (함수가 `query`를 안 쓰니까). 모델은 그걸 모르고 열심히 검색어를 넣어요 → 설명서와 실제 동작이 다르면 모델은 속을 수밖에 없다는 것도 보여 줘요

## 4단계: 설명서를 대충 쓰면?

smolagents는 설명서가 빠지면 **도구를 만드는 순간** 에러를 내요. 모델이 헷갈리기 전에 미리 막는 거예요.
- 4-A: docstring에 `Args:` 설명이 없음
- 4-B: 입력에 타입 힌트가 없음

`try` / `except` = "해 보고, 에러가 나면 멈추지 말고 에러 내용만 출력해" 라는 뜻이에요. 일부러 에러를 내는 칸이라 이렇게 감쌌어요.

In [ ]:
print("----- 4-A: Args 설명 없음 -----")
try:
    @tool
    def no_args_doc(query: str) -> str:
        """Returns the best catering service."""
        return "Gotham Catering Co."
except Exception as e:
    print(type(e).__name__, "→", e)

print("\n----- 4-B: 타입 힌트 없음 -----")
try:
    @tool
    def no_type_hint(query) -> str:
        """Returns the best catering service.

        Args:
            query: A search term.
        """
        return "Gotham Catering Co."
except Exception as e:
    print(type(e).__name__, "→", e)

✅ **확인할 것**
- 4-A: `DocstringParsingException → ... has no description for the argument 'query'` (query 설명이 없어요)
- 4-B: `TypeHintParsingException → Argument query is missing a type hint ...` (query가 무슨 종류인지 안 적었어요)
- 둘 다 **모델을 부르기도 전에** 나는 에러 = 설명서 4칸은 선택이 아니라 **필수**

## 5단계: `Tool` 클래스로 도구 만들기 (원문 예제 ②)

이번엔 함수 대신 **클래스**로 만들어요. 설명서 4칸을 **내가 직접** 적고, 실제 일은 `forward()`에 써요.

| 클래스에 적는 것 | 뜻 | `@tool`에서는 |
|---|---|---|
| `name` | 이름 | 함수 이름 |
| `description` | 설명 | docstring |
| `inputs` | 입력마다 `type`·`description` | 타입 힌트 + `Args:` |
| `output_type` | 출력 종류 | `-> str` |
| `forward(self, category)` | **실제로 일하는 부분** | 함수 본문 |

주의: `forward`의 인자 이름(`category`)은 `inputs`의 키(`"category"`)와 **똑같아야** 해요. 다르면 만들 때 에러가 나요.

In [ ]:
from smolagents import Tool


class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets.",
        }
        return themes.get(
            category.lower(),
            "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.",
        )


party_theme_tool = SuperheroPartyThemeTool()  # 클래스(설계도) → 실제 도구 하나 만들기

print("모델 없이 직접 불러 보기:", party_theme_tool("villain masquerade"))
print()

agent = CodeAgent(tools=[party_theme_tool], model=model, max_steps=4)
result = agent.run("What would be a good superhero party idea for a 'villain masquerade' theme?")
print("\n★ 최종 답:", result)

✅ **확인할 것**
- 직접 부르기: `Gotham Rogues' Ball: ...` (모델 없이 바로)
- 에이전트: 코드 상자에 `superhero_party_theme_generator(category="villain masquerade")` → 최종 답도 `Gotham Rogues' Ball: ...`

### 5-B: 목록에 없는 테마를 물어보면?

도구는 `"Themed party idea not found. Try ..."`를 돌려줘요. 이 말이 **관찰**로 모델에게 돌아가요. 모델이 어떻게 반응하는지 봐요.

In [ ]:
result = agent.run("What would be a good superhero party idea for a 'space pirates' theme?")
print("\n★ 최종 답:", result)

✅ **확인할 것** (작은 모델이라 매번 다를 수 있어요)
- 1바퀴: `category="space pirates"` → 관찰 `Themed party idea not found. Try ...`
- 그다음 모델이 고르는 길:
  - (가) 목록에 있는 값(`futuristic Gotham` 등)으로 **다시 시도**
  - (나) 도구 없이 **자기 머리로** 아이디어를 지어서 답
  - (다) "그런 테마는 없다"고 솔직하게 답
- 어느 쪽이든, 도구가 돌려준 **친절한 안내 문장**("Try ...")이 모델의 다음 행동을 이끈다는 점이 핵심이에요. 에러 메시지도 설명서의 일부예요

## 6단계: 같은 도구, 두 방법 → 모델이 보는 설명서 비교

2단계의 출장 요리 도구를 **클래스로 다시** 만들고, 모델이 읽는 모양(`to_code_prompt()`)을 나란히 찍어 봐요.

In [ ]:
class CateringServiceTool(Tool):
    name = "catering_service_tool"
    description = "This tool returns the highest-rated catering service in Gotham City."
    inputs = {"query": {"type": "string", "description": "A search term for finding catering services."}}
    output_type = "string"

    def forward(self, query: str):
        services = {"Gotham Catering Co.": 4.9, "Wayne Manor Catering": 4.8, "Gotham City Events": 4.7}
        return max(services, key=services.get)


print("===== @tool 버전 =====")
print(catering_service_tool.to_code_prompt())
print("\n===== Tool 클래스 버전 =====")
print(CateringServiceTool().to_code_prompt())

print("\n@tool로 만든 것도 Tool의 한 종류인가?", isinstance(catering_service_tool, Tool))

✅ **확인할 것**
- 두 출력이 **똑같아요** → 모델 입장에선 어떤 방법으로 만들었는지 구별 못 함
- `True` → `@tool`은 사실 **`Tool` 클래스를 자동으로 만들어 주는 지름길**이에요

**그럼 언제 클래스를 쓰나?** (AI 보충)
- 간단한 도구 → `@tool` (원문 추천)
- 도구가 **처음 한 번 준비할 게 있을 때**: 예) 큰 모델·데이터를 미리 불러 두기, 검색 결과 개수 같은 **설정값을 받아 두기**
  - 기본 도구 `VisitWebpageTool(max_output_length=4000)`처럼 괄호 안에 설정을 넣는 게 클래스라서 가능해요

## 7단계: 기본 도구 상자 + 에이전트가 나에게 질문하기

### 7-A: `add_base_tools=True`를 켜면 무엇이 들어오나

In [ ]:
from smolagents import DuckDuckGoSearchTool, ToolCallingAgent

code_agent = CodeAgent(tools=[], model=model, add_base_tools=True)
tc_agent = ToolCallingAgent(tools=[], model=model, add_base_tools=True)

print("CodeAgent        :", list(code_agent.tools))
print("ToolCallingAgent :", list(tc_agent.tools))

print("\n----- 이름이 같은 도구 두 개를 넣으면? -----")
try:
    CodeAgent(tools=[DuckDuckGoSearchTool(), DuckDuckGoSearchTool()], model=model)
except Exception as e:
    print(type(e).__name__, "→", e)

✅ **확인할 것**
- CodeAgent: `['web_search', 'visit_webpage', 'final_answer']`
- ToolCallingAgent: 여기에 **`python_interpreter`가 더** 있어요
  - 이유: CodeAgent는 행동 자체가 파이썬 코드라서 계산기 도구가 필요 없어요. ToolCallingAgent는 계산을 못 하니까(지난 섹션) 파이썬 실행 도구를 따로 줘요
- 이름이 같으면 `ValueError → Each tool ... should have a unique name!`
  - 원문은 Alfred가 `DuckDuckGoSearchTool`과 `GoogleSearchTool`을 **같이** 쓴다고 했는데, 둘 다 이름이 `web_search`라서 **그대로 넣으면 이 에러가 나요**. 같이 쓰려면 하나의 이름을 바꿔야 해요 (예: `google_tool.name = "google_search"`). `GoogleSearchTool`은 SerpAPI 같은 유료 검색 API 키도 필요해요

### 7-B: `UserInputTool` — 에이전트가 **나에게** 묻는 도구

기본 도구 중 재미있는 것이에요. 에이전트가 모르는 걸 **사람에게 물어봐요.**
실행하면 칸 아래에 **입력창**이 생겨요 → 숫자(예: `50`)를 치고 `Enter`

In [ ]:
from smolagents import UserInputTool

agent = CodeAgent(tools=[UserInputTool()], model=model, max_steps=4)

result = agent.run(
    "Ask the user how many guests are coming to the party. "
    "Each table seats 8 people. Tell me how many tables we need."
)
print("\n★ 최종 답:", result)

✅ **확인할 것**
- 입력창에 `How many guests ... => Type your answer here:` 같은 질문이 뜸 = 모델이 쓴 코드 `user_input(question="...")`가 실행된 것
- 내가 친 답이 **관찰**이 되고, 모델이 그걸로 계산해서 답함 (50명이면 50 ÷ 8 = 6.25 → 7개)
- 입력창이 안 보이면: 칸 아래로 스크롤. 멈춘 것 같으면 입력창에 답을 안 친 것

→ 도구는 인터넷·계산뿐 아니라 **사람**과도 연결될 수 있어요 (Unit 1 Agency 스펙트럼의 "사람에게 확인받기")

## 8단계: 남이 만든 도구 가져오기·내 도구 공유하기 (읽기만)

원문 후반부예요. **이 노트북에서는 실행하지 않아요.** 계정·키·추가 설치가 필요하거나, 남의 코드를 내 PC에서 돌려야 하거든요.

| 방법 | 코드 | 무엇 | 실행 안 하는 이유 |
|---|---|---|---|
| Hub에 올리기 | `party_theme_tool.push_to_hub("아이디/이름")` | 내 도구를 HF Hub에 공유 | HF 토큰(쓰기 권한) 필요. 공개됨 → 우리 규칙상 **승인 후** |
| Hub에서 가져오기 | `load_tool("m-ric/text-to-image", trust_remote_code=True)` | 남이 올린 도구 | `trust_remote_code=True` = **남의 코드를 내 PC에서 실행** → 승인 필요 + HF 크레딧 |
| Space를 도구로 | `Tool.from_space("black-forest-labs/FLUX.1-schnell", name=..., description=...)` | Gradio 앱(Space)을 도구처럼 | `gradio_client` 설치 필요, 이미지 생성은 HF GPU 사용량 제한 |
| LangChain 도구 | `Tool.from_langchain(...)` | 다른 프레임워크의 도구를 빌려 씀 | `langchain-community` 설치 + SerpAPI 키 필요 |
| MCP 서버 | `ToolCollection.from_mcp(서버설정, trust_remote_code=True)` | MCP 서버 하나의 **도구 여러 개를 한꺼번에** | `smolagents[mcp]` + `uvx` 설치 필요, 남의 서버 코드 실행 |

공통점: 어떤 방법이든 결과물은 **2~6단계에서 만든 것과 같은 `Tool`**이에요. 이름·설명·입력·출력이 있는 함수라서, `tools=[...]` 목록에 넣기만 하면 돼요.